# Notebook 01 — Data Loading & QA Dataset Creation

This notebook:
1. Downloads the Indiana University CXR dataset via Kaggle (images + CSV reports)
2. Parses the CSV reports (no XML parsing needed)
3. Generates a QA dataset using Groq LLaMA 3.1 8B Instant

**Before running:** Add these 4 keys to Colab Secrets (🔑 icon in left sidebar):
- `KAGGLE_USERNAME`, `KAGGLE_KEY` — from kaggle.com/settings → API → Create New Token
- `GROQ_API_KEY` — from console.groq.com
- `HF_TOKEN` — from huggingface.co/settings/tokens

In [1]:
!pip install -q groq tqdm pandas kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 4.5 MB/s eta 0:00:00


In [3]:
import os, sys
from google.colab import drive, userdata
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/cxr_rag'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive mounted at', DRIVE_ROOT)

Mounted at /content/drive
Drive mounted at /content/drive/MyDrive/cxr_rag


In [4]:
# ── Kaggle credentials ────────────────────────────────────────────────────────
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
print('Kaggle credentials set.')

Kaggle credentials set.


In [5]:
# ── Download Indiana University CXR via Kaggle ────────────────────────────────
# Downloads: images/ folder + indiana_reports.csv + indiana_projections.csv
# Total size: ~1 GB

KAGGLE_DIR = '/content/openi'

if not os.path.exists(os.path.join(KAGGLE_DIR, 'indiana_reports.csv')):
    print('Downloading dataset (~1 GB) ...')
    import subprocess
    os.makedirs(KAGGLE_DIR, exist_ok=True)
    subprocess.run([
        'kaggle', 'datasets', 'download',
        '-d', 'raddar/chest-xrays-indiana-university',
        '-p', KAGGLE_DIR,
        '--unzip'
    ], check=True)
    print('Done.')
else:
    print('Dataset already downloaded.')

print('\nContents of', KAGGLE_DIR + ':')
for f in os.listdir(KAGGLE_DIR):
    print(' ', f)

Done.

Contents of /content/openi:
  images
  indiana_reports.csv
  indiana_projections.csv


In [6]:
# ── Auto-detect image directory ───────────────────────────────────────────────
import glob

png_files = glob.glob(os.path.join(KAGGLE_DIR, '**', '*.png'), recursive=True)
if not png_files:
    raise RuntimeError('No PNG images found. Check the contents of ' + KAGGLE_DIR)

IMAGES_DIR = os.path.dirname(png_files[0])
print(f'Found {len(png_files)} PNG images in: {IMAGES_DIR}')

Found 7470 PNG images in: /content/openi/images/images_normalized


In [7]:
# ── Clone project repo ────────────────────────────────────────────────────────
REPO_URL = 'https://github.com/mohamedtaha77/cxr-rag-system.git'

if not os.path.exists('/content/cxr-rag-system'):
    !git clone {REPO_URL} /content/cxr-rag-system
else:
    !git -C /content/cxr-rag-system pull

sys.path.insert(0, '/content/cxr-rag-system')
print('Repo ready.')

Cloning into '/content/cxr-rag-system'...
remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 42 (delta 12), reused 31 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (42/42), 33.26 KiB | 4.75 MiB/s, done.
Resolving deltas: 100% (12/12), done.
Repo ready.


In [8]:
# ── Load dataset from Kaggle CSVs ─────────────────────────────────────────────
# Uses indiana_reports.csv + indiana_projections.csv
# Much simpler than XML parsing — projections CSV tells us which images are frontal

from src.data.openi_loader import OpenILoader

loader = OpenILoader(images_dir=IMAGES_DIR)
df = loader.load_from_kaggle_csvs(kaggle_dir=KAGGLE_DIR)

print(f'Loaded {len(df)} studies with impression + frontal image')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

Loaded 3652 studies with impression + frontal image
Columns: ['study_id', 'impression', 'findings', 'image_path']


,study_id,impression,findings,image_path
0,1,Normal chest x-XXXX.,The cardiac silhouette and mediastinum size ar...,/content/openi/images/images_normalized/1_IM-0...
1,2,No acute pulmonary findings.,Borderline cardiomegaly. Midline sternotomy XX...,/content/openi/images/images_normalized/2_IM-0...
2,3,"No displaced rib fractures, pneumothorax, or p...",,/content/openi/images/images_normalized/3_IM-1...


In [9]:
# ── Train / val / test split ──────────────────────────────────────────────────
import pandas as pd, shutil

train_df, val_df, test_df = loader.train_val_test_split(df)
full_df = pd.concat([train_df, val_df, test_df])

os.makedirs('/content/cxr-rag-system/data/processed', exist_ok=True)
full_df.to_csv('/content/cxr-rag-system/data/processed/reports_corpus.csv', index=False)
shutil.copy('/content/cxr-rag-system/data/processed/reports_corpus.csv',
            os.path.join(DRIVE_ROOT, 'reports_corpus.csv'))

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print('Sample impressions:')
for imp in full_df['impression'].head(3):
    print(' -', imp[:120])

Train: 2921 | Val: 365 | Test: 366
Sample impressions:
 - 1. No evidence of active disease.
 - Heart size is normal lungs are clear. No evidence of tuberculosis. Minimal scoliosis.
 - 1. Interval enlargement of right middle lobe mass, highly suspicious for malignancy. Recommend CT of the chest/abdomen w


In [10]:
# ── Configure Groq ────────────────────────────────────────────────────────────
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

from src.data.qa_creator import QACreator
creator = QACreator(groq_api_key=GROQ_API_KEY)
print('QA creator ready.')

QA creator ready.


In [12]:
# ── Generate QA dataset ───────────────────────────────────────────────────────
# max_studies=500  → ~3,000 pairs, ~30 min
# max_studies=None → full dataset (~23,000 pairs, ~3 hrs)

QA_OUTPUT = '/content/cxr-rag-system/data/processed/qa_dataset.jsonl'

pairs = creator.generate_dataset(
    df=full_df,
    output_path=QA_OUTPUT,
    max_studies=200,
)

print(f'Generated {len(pairs)} QA pairs')
shutil.copy(QA_OUTPUT, os.path.join(DRIVE_ROOT, 'qa_dataset.jsonl'))

Generating QA pairs: 100%|██████████| 200/200 [1:00:06<00:00, 18.03s/it]

Generated 1515 QA pairs → /content/cxr-rag-system/data/processed/qa_dataset.jsonl
Generated 1515 QA pairs


'/content/drive/MyDrive/cxr_rag/qa_dataset.jsonl'

In [13]:
# ── Inspect sample QA pairs ───────────────────────────────────────────────────
import json

with open(QA_OUTPUT) as f:
    samples = [json.loads(l) for l in f][:5]

for s in samples:
    print(f"Category : {s['category']}")
    print(f"Q        : {s['question']}")
    print(f"A        : {s['answer']}")
    print()

Category : Consolidation
Q        : Is there consolidation visible in this chest X-ray?
A        : No consolidation is observed in the lungs.

Category : Consolidation
Q        : Can pulmonary consolidation be identified?
A        : No, pulmonary consolidation is not observed. The lungs are free of focal airspace disease.

Category : Consolidation
Q        : Are there signs of airspace consolidation?
A        : No, airspace consolidation is not observed. The lungs are free of focal airspace disease.

Category : No Finding
Q        : Is this chest X-ray normal?
A        : The radiograph appears to be essentially normal, with no evidence of active disease, normal heart size and pulmonary vascularity, and no focal lung or pleural abnormalities.

Category : No Finding
Q        : Are there any abnormal findings in this radiograph?
A        : There are no abnormal findings in the lungs, heart, or pleura. However, degenerative changes are present in the spine.



In [14]:
# ── Dataset statistics ────────────────────────────────────────────────────────
qa_df = pd.read_json(QA_OUTPUT, lines=True)
print(f'Total pairs   : {len(qa_df)}')
print(f'Unique studies: {qa_df["study_id"].nunique()}')
print(f'\nSplit distribution:')
print(qa_df['split'].value_counts())
print(f'\nCategory distribution:')
print(qa_df['category'].value_counts())

Total pairs   : 1515
Unique studies: 200

Split distribution:
split
train    1515
Name: count, dtype: int64

Category distribution:
category
No Finding              446
Consolidation           281
Pleural Effusion        189
Lung Lesion             118
Lung Opacity            109
Enlarged Mediastinum     89
Atelectasis              78
Cardiomegaly             54
Support Devices          45
Pneumothorax             39
Edema                    37
Fracture                 16
Pneumonia                12
Pleural Other             2
Name: count, dtype: int64
